# PyTorch nn.Module Practice

**Run locally** — this notebook requires PyTorch.

This notebook gives you hands-on practice building RL networks with `nn.Module`.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt


## 1. Build a QNetwork

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, n_actions, hidden_dim=64):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, n_actions)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

q_net = QNetwork(state_dim=4, n_actions=2)
state = torch.FloatTensor([[0.1, -0.2, 0.05, 0.3]])
q_vals = q_net(state)
print('Q-values:', q_vals.detach().numpy())
print('Best action:', q_vals.argmax(dim=1).item())
print('Parameters:', sum(p.numel() for p in q_net.parameters()))


## 2. PolicyNetwork with softmax output

In [ ]:
class PolicyNetwork(nn.Module):
    def __init__(self, state_dim, n_actions, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_actions)
        )

    def forward(self, x):
        logits = self.net(x)
        return F.softmax(logits, dim=-1)

    def get_action(self, state):
        probs = self.forward(state)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        return action.item(), dist.log_prob(action)

pi_net = PolicyNetwork(state_dim=4, n_actions=3)
state = torch.FloatTensor([[0.1, -0.2, 0.05, 0.3]])
probs = pi_net(state)
print('Action probs:', probs.detach().numpy())
print('Sum:', probs.sum().item())
action, log_prob = pi_net.get_action(state)
print(f'Sampled action: {action}, log_prob: {log_prob.item():.4f}')


## 3. Training step — DQN style

In [ ]:
model = QNetwork(state_dim=4, n_actions=2)
optimizer = optim.Adam(model.parameters(), lr=3e-4)

def train_step(states_np, actions_np, targets_np):
    states = torch.FloatTensor(states_np)
    actions = torch.LongTensor(actions_np)
    targets = torch.FloatTensor(targets_np)

    optimizer.zero_grad()
    q_vals = model(states)
    q_selected = q_vals.gather(1, actions.unsqueeze(1)).squeeze(1)
    loss = F.mse_loss(q_selected, targets)
    loss.backward()
    optimizer.step()
    return loss.item()

# Train on random data
np.random.seed(42)
losses = []
for _ in range(200):
    states = np.random.randn(32, 4).astype(np.float32)
    actions = np.random.randint(0, 2, 32)
    targets = np.random.randn(32).astype(np.float32)
    loss = train_step(states, actions, targets)
    losses.append(loss)

plt.plot(losses)
plt.xlabel('Step'); plt.ylabel('TD Loss'); plt.title('DQN Training Step')
plt.show()


## 4. Target Network
A frozen copy updated every C steps.

In [ ]:
import copy

online_net = QNetwork(state_dim=4, n_actions=2)
target_net = copy.deepcopy(online_net)

# Freeze target network
for p in target_net.parameters():
    p.requires_grad = False

C = 10  # update target every 10 steps
optimizer = optim.Adam(online_net.parameters(), lr=3e-4)

for step in range(50):
    states = torch.FloatTensor(np.random.randn(8, 4))
    actions = torch.LongTensor(np.random.randint(0, 2, 8))
    rewards = torch.FloatTensor(np.random.randn(8))
    next_states = torch.FloatTensor(np.random.randn(8, 4))

    with torch.no_grad():
        next_q = target_net(next_states).max(dim=1)[0]
        targets = rewards + 0.99 * next_q

    optimizer.zero_grad()
    q_vals = online_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
    loss = F.mse_loss(q_vals, targets)
    loss.backward()
    optimizer.step()

    if (step + 1) % C == 0:
        target_net.load_state_dict(online_net.state_dict())
        print(f'Step {step+1}: updated target network')
